In [5]:
import tqdm as notebook_tqdm
from datasets import Dataset
import pandas as pd
import math
import torch
from transformers import XLNetTokenizer, XLNetLMHeadModel, Trainer, TrainingArguments, pipeline

In [6]:
df = pd.read_csv("final_dataset.csv")

# Convert to HuggingFace Dataset using only the Conversation column
dataset = Dataset.from_pandas(df[["Conversation"]].rename(columns={"Conversation": "text"}))
dataset = {"train": dataset}

In [7]:
# Step 3: Load XLNet tokenizer and model

tokenizer = XLNetTokenizer.from_pretrained("xlnet-base-cased")
model = XLNetLMHeadModel.from_pretrained("xlnet-base-cased")

In [8]:
print(dataset["train"])

Dataset({
    features: ['text'],
    num_rows: 300000
})


In [9]:
def tokenize_function(example):
    return tokenizer(example["text"])

tokenized_dataset = dataset["train"].map(tokenize_function, batched=True)

Map: 100%|██████████| 300000/300000 [00:32<00:00, 9202.45 examples/s]


In [10]:
# Grouping function
block_size = 128

def group_texts(examples):
    concatenated = []
    for input_ids in examples["input_ids"]:
        concatenated.extend(input_ids)

    total_length = (len(concatenated) // block_size) * block_size
    input_ids = [concatenated[i : i + block_size] for i in range(0, total_length, block_size)]
    attention_mask = [[1] * block_size for _ in input_ids]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": input_ids.copy(),
    }

In [11]:
# Step 4: Training in 10k chunks with evaluation
chunk_size = 10000
num_chunks = math.ceil(len(df) / chunk_size)
val_accuracies = []

for chunk_idx in range(num_chunks):
    print(f"Training chunk {chunk_idx + 1}/{num_chunks}...")

    chunk_df = df.iloc[chunk_idx * chunk_size : (chunk_idx + 1) * chunk_size]
    dataset = Dataset.from_pandas(chunk_df[["Conversation"]].rename(columns={"Conversation": "text"}))
    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    lm_dataset = tokenized_dataset.map(group_texts, batched=True, remove_columns=tokenized_dataset.column_names)

    # Split into train and validation
    split_idx = int(0.9 * len(lm_dataset))
    train_dataset = lm_dataset.select(range(split_idx))
    eval_dataset = lm_dataset.select(range(split_idx, len(lm_dataset)))

    training_args = TrainingArguments(
        output_dir=f"temp/xlnet-hinglish-chunk{chunk_idx + 1}",
        evaluation_strategy="epoch",
        num_train_epochs=3,
        per_device_train_batch_size=8,
        save_steps=500,
        save_total_limit=2,
        logging_steps=100,
        warmup_steps=100,
        weight_decay=0.01,
        fp16=True,
        overwrite_output_dir=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
    )

    trainer.train()

    # Evaluate and store accuracy
    eval_result = trainer.evaluate()
    if "eval_loss" in eval_result:
        val_accuracy = 1 - eval_result["eval_loss"]
        val_accuracies.append(val_accuracy)

    # Save intermediate model
    model.save_pretrained(f"temp/xlnet-hinglish-chunk{chunk_idx + 1}")
    tokenizer.save_pretrained(f"temp/xlnet-hinglish-chunk{chunk_idx + 1}")

# Step 5: Final model save
model.save_pretrained("./xlnet-hinglish-final")
tokenizer.save_pretrained("./xlnet-hinglish-final")

Training chunk 1/30...


Map: 100%|██████████| 10000/10000 [00:00<00:00, 52147.85 examples/s]
/home/shady/Desktop/project/venv/lib64/python3.10/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_13788/1751390237.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,0.000700,0.000113
2,0.000500,0.000029
3,0.000200,0.000016


Training chunk 2/30...


Map: 100%|██████████| 10000/10000 [00:00<00:00, 45396.14 examples/s]
/home/shady/Desktop/project/venv/lib64/python3.10/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_13788/1751390237.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


...

Epoch,Training Loss,Validation Loss
1,0.000100,0.000000
2,0.000000,0.000017
3,0.000000,0.000000


Training chunk 30/30...


Map: 100%|██████████| 10000/10000 [00:00<00:00, 46004.50 examples/s]
/home/shady/Desktop/project/venv/lib64/python3.10/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_13788/1751390237.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,0.001300,0.000001
2,0.000000,0.000000
3,0.000000,0.000000


('./xlnet-hinglish-final/tokenizer_config.json',
 './xlnet-hinglish-final/special_tokens_map.json',
 './xlnet-hinglish-final/spiece.model',
 './xlnet-hinglish-final/added_tokens.json')

In [ ]:
generator = pipeline("text-generation", model="./xlnet-hinglish-final", tokenizer=tokenizer)

user_input = input("Enter a prompt: ")
print(f"testing output for the input: {user_input}")
print(generator(user_input))

Device set to use cuda:0


testing output for the input: mai
[{'generated_text': 'mai'}]


: 